# Structured Output
<img src="./assets/LC_StructuredOutput.png" width="500">


## Setup

Load and/or check for needed environmental variables

In [4]:
from dotenv import load_dotenv

from env_utils import doublecheck_env

load_dotenv()

# Check and print results
doublecheck_env("example.env")

OPENAI_API_KEY=****ywsA
DEEPSEEK_API_KEY=****c1e4
DEEPSEEK_BASE_URL=****.com
LANGSMITH_API_KEY=****b63c
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=****ials


## Structured Output Example

In [5]:
from typing_extensions import TypedDict

from langchain.agents import create_agent


# 定义结构化输出的目标格式
# 这里要求 agent 最后返回 name、email、phone 三个字段
class ContactInfo(TypedDict):
    name: str
    email: str
    phone: str


# 创建 agent
# response_format=ContactInfo 表示:
# 不只是让模型输出普通文本,而是要求它按 ContactInfo 这套结构返回结果
#并且实际上是将ContactInfo包装为工具自动加入到tool列表中
#最后强制模型使用这个工具返回结构化结果
#不过这里好像是因为deepseek不支持结构化输出?
#
agent = create_agent(
    model="deepseek-chat",
    response_format=ContactInfo,
)

# 这里是一段待提取信息的对话文本
# 目标是让模型从自然语言里整理出联系人姓名、邮箱和电话
recorded_conversation = """We talked with John Doe. He works over at Example. His number is, let's see,
five, five, five, one two three, four, five, six seven. Did you get that?
And, his email was john at example.com. He wanted to order 50 boxes of Captain Crunch."""

# 调用 agent
# messages 里放用户输入的文本内容
result = agent.invoke(
    {"messages": [{"role": "user", "content": recorded_conversation}]}
)
#看一下原始输出
print(result)
# structured_response 是结构化提取后的最终结果
# 一般会得到类似:
# {'name': 'John Doe', 'email': 'john@example.com', 'phone': '5551234567'}
result["structured_response"]


#原始输出
# {
#     'messages': [
#         HumanMessage(
#             content="We talked with John Doe. He works over at Example. His number is, let's see,\nfive, five, five, one two three, four, five, six seven. Did you get that?\nAnd, his email was john at example.com. He wanted to order 50 boxes of Captain Crunch.用中文回答",
#             additional_kwargs={},
#             response_metadata={},
#             id='5999cc7d-618f-4935-94ee-639e284940a3'
#         ),
#         AIMessage(
#             content='',
#             additional_kwargs={
#                 'refusal': None
#             },
#             response_metadata={
#                 'token_usage': {
#                     'completion_tokens': 75,
#                     'prompt_tokens': 388,
#                     'total_tokens': 463,
#                     'completion_tokens_details': None,
#                     'prompt_tokens_details': {
#                         'audio_tokens': None,
#                         'cached_tokens': 320
#                     },
#                     'prompt_cache_hit_tokens': 320,
#                     'prompt_cache_miss_tokens': 68
#                 },
#                 'model_provider': 'deepseek',
#                 'model_name': 'deepseek-chat',
#                 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache_new_kvcache',
#                 'id': '396a32e1-f015-4bb4-af99-038dcfd829af',
#                 'finish_reason': 'tool_calls',
#                 'logprobs': None
#             },
#             id='lc_run--749b0870-f9f3-4acf-8bef-0b5e3940dd78-0',
#             tool_calls=[
#                 {
#                     'name': 'ContactInfo',
#                     'args': {
#                         'name': 'John Doe',
#                         'email': 'john@example.com',
#                         'phone': '5551234567'
#                     },
#                     'id': 'call_00_qKMUqZ28cNoB2Pl6Rlvftw3M',
#                     'type': 'tool_call'
#                 }
#             ],
#             usage_metadata={
#                 'input_tokens': 388,
#                 'output_tokens': 75,
#                 'total_tokens': 463,
#                 'input_token_details': {
#                     'cache_read': 320
#                 },
#                 'output_token_details': {}
#             }
#         ),
#         ToolMessage(
#             content="Returning structured response: {'name': 'John Doe', 'email': 'john@example.com', 'phone': '5551234567'}",
#             name='ContactInfo',
#             id='2a024e00-06a5-4c8e-994d-7cde496be3af',
#             tool_call_id='call_00_qKMUqZ28cNoB2Pl6Rlvftw3M'
#         )
#     ],
#     'structured_response': {
#         'name': 'John Doe',
#         'email': 'john@example.com',
#         'phone': '5551234567'
#     }
# }

{'messages': [HumanMessage(content="We talked with John Doe. He works over at Example. His number is, let's see,\nfive, five, five, one two three, four, five, six seven. Did you get that?\nAnd, his email was john at example.com. He wanted to order 50 boxes of Captain Crunch.", additional_kwargs={}, response_metadata={}, id='5b0c7685-3aec-4ac8-a7ec-f5e64639ebe4'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 77, 'prompt_tokens': 385, 'total_tokens': 462, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256}, 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 129}, 'model_provider': 'deepseek', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache_new_kvcache', 'id': '15e12e10-5699-4b49-b008-c03c17ccf33c', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--b973fc95-4c90-42fe-99ef-6312559ca4ff-0', tool_calls=[{'name

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '555-123-4567'}

#### Multiple data types are supported 

* pydantic `BaseModel`
* `TypedDict`
* `dataclasses`
* json schema (dict)

In [3]:
from langchain.agents import create_agent
from pydantic import BaseModel


#这里主要演示各种数据类型都能够起到结构化输出的效果
#不过最多的还是BaseModel?(或许)
class ContactInfo(BaseModel):
    name: str
    email: str
    phone: str


agent = create_agent(model="deepseek-chat", response_format=ContactInfo)

recorded_conversation = """ We talked with John Doe. He works over at Example. His number is, let's see, 
five, five, five, one two three, four, five, six seven. Did you get that?
And, his email was john at example.com. He wanted to order 50 boxes of Captain Crunch."""

result = agent.invoke(
    {"messages": [{"role": "user", "content": recorded_conversation}]}
)

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='5551234567')

In [ ]:
from decimal import Decimal, ROUND_HALF_UP
from typing import Optional

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from pydantic import BaseModel, ConfigDict, Field, PositiveInt

# 这个小 demo 同时复习工具调用、记忆和结构化输出
load_dotenv(".env")


class LoanAdviceResult(BaseModel):
    user_goal: str = Field(description="用户意图")
    loan_amount: Optional[float] = Field(default=None, description="贷款本金，单位：元")
    annual_interest_rate: Optional[float] = Field(default=None, description="年利率，小数形式，例如 0.032")
    loan_years: Optional[PositiveInt] = Field(default=None, description="贷款年限，单位：年")
    monthly_payment: Optional[float] = Field(default=None, description="预测月供，单位：元")
    total_payment: Optional[float] = Field(default=None, description="总还款额，单位：元")
    total_interest: Optional[float] = Field(default=None, description="总利息，单位：元")
    missing_info: list[str] = Field(default_factory=list, description="当前还缺少的信息")
    advice: str = Field(description="给用户的简短建议")
    follow_up_question: Optional[str] = Field(default=None, description="如果信息不完整，下一步追问")


class MonthlyPaymentInput(BaseModel):
    model_config = ConfigDict(extra="forbid")

    loan_amount: Decimal = Field(gt=0, description="贷款本金，单位元")
    annual_interest_rate: Decimal = Field(gt=0, description="年利率，小数形式，例如 0.032")
    loan_years: PositiveInt = Field(description="贷款年限，单位年")


def _calculate_monthly_payment(
        loan_amount: Decimal,
        annual_interest_rate: Decimal,
        loan_years: PositiveInt,
)->dict:
    #转换为月
    months=loan_years*12
    #转换为月利率
    monthly_rate=annual_interest_rate/12
    #由于输入时规定了利率大于0,因此不考虑利率为0的情况
    #接下来就是按照公式计算月还款额=贷款数*月利率*(1+月利率)^月数/((1+月利率)^月数-1)
    #先计算复利因子(1+月利率)^月数
    x=(Decimal("1")+monthly_rate)**months
    monthly_payment=loan_amount*monthly_rate*x/(x-1)
    #最后返回字典包含月还款额,总还款额以及总利息
    #总偿还金额
    total_payment=monthly_payment*months
    #总利息
    total_interest=monthly_payment*months-loan_amount
    return{
        #这里使用quantize来限制小数位数,并且使用的是四舍五入
        "monthly_payment": monthly_payment.quantize(Decimal("0.01"),rounding=ROUND_HALF_UP),
        "total_payment": total_payment.quantize(Decimal("0.01"),rounding=ROUND_HALF_UP),
        "total_interest": total_interest.quantize(Decimal("0.01"),rounding=ROUND_HALF_UP),
    }


@tool(args_schema=MonthlyPaymentInput)
def calculate_monthly_payment(
    loan_amount: Decimal,
    annual_interest_rate: Decimal,
    loan_years: PositiveInt,
) -> str:
    """计算等额本息的月供、总还款额和总利息。"""
    result = _calculate_monthly_payment(loan_amount, annual_interest_rate, loan_years)
    return (
        f"monthly_payment: {result['monthly_payment']}\n"
        f"total_payment: {result['total_payment']}\n"
        f"total_interest: {result['total_interest']}"
    )


LOAN_STRUCTURED_PROMPT = """
你是一名贷款顾问助手。

要求：
1. 结合同一线程里的历史对话，复用用户之前提供的信息。
2. 只要涉及月供、总还款额、总利息的精确计算，必须调用工具。
3. 如果信息不足，不要猜测，不要补造数据。
4. 最终必须输出结构化结果。
5. 如果信息不完整，把缺失字段写入 missing_info，并给出 follow_up_question。
6. 回答默认使用中文，简洁清楚。
"""


agent = create_agent(
    model="deepseek-chat",
    response_format=LoanAdviceResult,
    tools=[calculate_monthly_payment],
    checkpointer=InMemorySaver(),
    system_prompt=LOAN_STRUCTURED_PROMPT
)

In [ ]:
for token,metadata in agent.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": "你好,我要贷款"
            }
        ]
    },
    {
        "configurable":{
            "thread_id":1
        }
    },
        stream_mode="messages"

    ):
        print(token.content,end="")

for _ in range(3):
    User_Input=input()
    print(User_Input)
    for token,metadata in agent.stream(
        {"messages": [{"role": "user", "content": User_Input}]},
        {
        "configurable":{
            "thread_id":1
        }
    },
        stream_mode="messages"
    ):
        print(token.content,end="")